# MovieLens EDA

Before building anything, let's actually look at what we're working with. Jumping straight into model code without understanding the data is how you end up debugging weird edge cases for hours.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

print('movies:', movies.shape)
print('ratings:', ratings.shape)
movies.head()

In [ ]:
ratings.head(10)

## Basic stats

In [ ]:
print(f"Total ratings:  {len(ratings):,}")
print(f"Unique users:   {ratings['userId'].nunique():,}")
print(f"Unique movies:  {ratings['movieId'].nunique():,}")
print(f"Avg rating:     {ratings['rating'].mean():.2f}")
print(f"Sparsity:       {1 - len(ratings) / (ratings['userId'].nunique() * ratings['movieId'].nunique()):.2%}")

The matrix is ~98% empty. That's typical for collaborative filtering — most users haven't rated most movies. This is exactly the problem we're trying to solve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# rating distribution
ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# ratings per user
ratings_per_user = ratings.groupby('userId').size()
axes[1].hist(ratings_per_user, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Ratings per User')
axes[1].set_xlabel('Number of ratings')
axes[1].set_ylabel('Number of users')

plt.tight_layout()
plt.show()

print(f"Median ratings per user: {ratings_per_user.median():.0f}")
print(f"Max ratings by one user: {ratings_per_user.max()}")

In [ ]:
# most rated movies — these will dominate recommendations for sparse users
top_movies = (
    ratings.groupby('movieId')
    .size()
    .reset_index(name='num_ratings')
    .merge(movies[['movieId', 'title']], on='movieId')
    .sort_values('num_ratings', ascending=False)
    .head(15)
)

plt.figure(figsize=(10, 5))
plt.barh(top_movies['title'], top_movies['num_ratings'], color='steelblue')
plt.gca().invert_yaxis()
plt.title('Most Rated Movies')
plt.xlabel('Number of ratings')
plt.tight_layout()
plt.show()

In [ ]:
# average rating by movie (only for movies with enough ratings to be meaningful)
movie_stats = (
    ratings.groupby('movieId')
    .agg(avg_rating=('rating', 'mean'), num_ratings=('rating', 'count'))
    .reset_index()
    .merge(movies[['movieId', 'title']], on='movieId')
)

# filter to movies with at least 50 ratings so the average actually means something
well_rated = movie_stats[movie_stats['num_ratings'] >= 50].sort_values('avg_rating', ascending=False).head(15)

plt.figure(figsize=(10, 5))
plt.barh(well_rated['title'], well_rated['avg_rating'], color='coral')
plt.gca().invert_yaxis()
plt.title('Highest Rated Movies (min 50 ratings)')
plt.xlabel('Average Rating')
plt.xlim(3.5, 5)
plt.tight_layout()
plt.show()

In [ ]:
# genre breakdown
from collections import Counter

all_genres = [g for genres in movies['genres'].str.split('|') for g in genres if g != '(no genres listed)']
genre_counts = Counter(all_genres)

genres_df = pd.DataFrame(genre_counts.most_common(15), columns=['genre', 'count'])

plt.figure(figsize=(10, 5))
plt.barh(genres_df['genre'], genres_df['count'], color='mediumseagreen')
plt.gca().invert_yaxis()
plt.title('Movies by Genre')
plt.tight_layout()
plt.show()

## Key takeaways before modeling

- The matrix is ~98% sparse — most user-movie pairs have no rating. `fillna(0)` will be our approach.
- Rating distribution skews positive (people tend to rate movies they chose to watch, which creates selection bias).
- Some users have rated hundreds of movies, others just 20. The similarity scores will be more meaningful for power users.
- Cold start is a real problem: new users with few ratings will get noisy recommendations. We acknowledge this in the README.